# Experiment 7 – Transfer Learning for Image Classification

### Deep Learning Laboratory

**Aim:** To implement the concept of Transfer Learning for image classification using a pre-trained CNN model in TensorFlow/Keras.

### Learning Objectives
- Understand the concept of Transfer Learning.
- Understand pre-trained CNN models.
- Use a pre-trained model as a feature extractor.
- Add a new classification layer for a new dataset.
- Train and evaluate a Transfer Learning model.


## 1. Theory

**Transfer Learning** is a technique in which knowledge learned by a neural network on one large dataset is reused for a new but related task.

For example, a CNN trained on a large image dataset has already learned useful features such as edges, shapes, textures and patterns. Instead of training a complete CNN from the beginning, we can reuse these learned features and train only a new classifier for our problem.

### Basic Flow

```text
Pre-trained CNN
      ↓
Remove Original Classifier
      ↓
Reuse Learned Features
      ↓
Add New Classifier
      ↓
Train on New Dataset
      ↓
Image Classification
```

In this experiment, **MobileNetV2**, a pre-trained CNN available in Keras, is used. The model is trained on ImageNet. We use its convolutional base as a feature extractor and add a new classifier for two classes: **cats and dogs**.

In [ ]:
# Step 1: Import required libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

## 2. Load the Image Dataset

We use TensorFlow's `cats_vs_dogs` dataset from TensorFlow Datasets (TFDS).

Each image is resized to **160 × 160** pixels. A small subset is used so that the experiment can run reasonably quickly in a Colab lab session.

In [ ]:
# Step 2: Install TensorFlow Datasets if required
!pip -q install tensorflow-datasets

In [ ]:
# Step 3: Import TensorFlow Datasets and load Cats vs Dogs
import tensorflow_datasets as tfds

(ds_train, ds_test), ds_info = tfds.load(
    "cats_vs_dogs",
    split=["train[:80%]", "train[80%:90%]"],
    as_supervised=True,
    with_info=True
)

print("Dataset loaded successfully.")
print("Number of classes:", ds_info.features["label"].num_classes)

## 3. Preprocess the Images

Transfer learning models expect images in a suitable input size and pixel range. MobileNetV2 uses an input size such as 160 × 160 × 3 in this experiment.

We resize every image and convert its pixel values to floating-point values.

In [ ]:
# Step 4: Preprocessing function
IMG_SIZE = (160, 160)
BATCH_SIZE = 32


def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label

train_ds = ds_train.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = ds_test.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Preprocessing completed.")

In [ ]:
# Step 5: Display sample images
plt.figure(figsize=(10, 6))
for images, labels in train_ds.take(1):
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8))
        plt.title("Dog" if int(labels[i]) == 1 else "Cat")
        plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Load the Pre-trained MobileNetV2 Model

The ImageNet-trained MobileNetV2 model is used without its original classification layer.

`include_top=False` removes the original ImageNet classifier. The remaining layers act as a **feature extractor**.

The base model is frozen initially, so its pre-trained weights are not changed during the first training stage.

In [ ]:
# Step 6: Load MobileNetV2 without its original classifier
base_model = keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

print("Pre-trained MobileNetV2 loaded.")
print("Trainable parameters in base model:", base_model.trainable)

In [ ]:
# Step 7: Build the Transfer Learning model
inputs = keras.Input(shape=(160, 160, 3))

# MobileNetV2 preprocessing
x = keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

### Why do we use `GlobalAveragePooling2D`?

It converts the feature maps produced by MobileNetV2 into a smaller vector. This vector is then given to the new classification layer.

### Why is `sigmoid` used?

There are only two classes: cat and dog. Therefore, a single sigmoid output is sufficient for binary classification.

In [ ]:
# Step 8: Train the new classifier while keeping MobileNetV2 frozen
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=3
)

In [ ]:
# Step 9: Evaluate the Transfer Learning model
loss, accuracy = model.evaluate(test_ds, verbose=0)

print("Test Loss:", round(float(loss), 4))
print("Test Accuracy:", round(float(accuracy * 100), 2), "%")

In [ ]:
# Step 10: Display predictions
for images, labels in test_ds.take(1):
    probabilities = model.predict(images, verbose=0).flatten()

    plt.figure(figsize=(10, 6))
    for i in range(min(8, len(images))):
        predicted = "Dog" if probabilities[i] >= 0.5 else "Cat"
        actual = "Dog" if int(labels[i]) == 1 else "Cat"

        plt.subplot(2, 4, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8))
        plt.title(f"Actual: {actual}\nPredicted: {predicted}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Step 11: Plot training and validation accuracy
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Transfer Learning Accuracy")
plt.legend()
plt.show()

## 5. Optional Fine-Tuning Stage

After training the new classifier, some of the deeper layers of the pre-trained model can be unfrozen and trained with a **very small learning rate**. This is called fine-tuning.

Fine-tuning should be done carefully because a large learning rate may damage the useful pre-trained features.

In [ ]:
# Step 12: Optional fine-tuning of the last few layers
base_model.trainable = True

# Freeze most layers and fine-tune only the last 20 layers
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

fine_tune_history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=2
)

fine_loss, fine_accuracy = model.evaluate(test_ds, verbose=0)
print("Fine-tuned Test Accuracy:", round(float(fine_accuracy * 100), 2), "%")

## 6. Student Practice

Try the following changes:

1. Change the number of frozen MobileNetV2 layers.
2. Change the dropout rate from `0.2` to `0.4`.
3. Change the learning rate of the classifier.
4. Train for more epochs and compare validation accuracy.
5. Compare the accuracy before and after fine-tuning.

**Observation:** Record the configuration that gives the best validation/test accuracy.

## Result

Thus, the concept of **Transfer Learning** was successfully implemented for image classification using the pre-trained MobileNetV2 model in TensorFlow/Keras. The learned features of the pre-trained model were reused for a new binary image-classification task.

## Viva Questions

1. What is Transfer Learning?
2. Why is Transfer Learning useful?
3. What is a pre-trained model?
4. What is MobileNetV2?
5. What does `include_top=False` mean?
6. Why do we initially freeze the base model?
7. What is feature extraction?
8. What is fine-tuning?
9. Why is a very small learning rate used during fine-tuning?
10. Why is sigmoid used in this experiment?